# AIM:
create evaluation workflow. Taking the manually extracted Ci impacts (validation set) and compare it with the CI impacts (llm_geolocations.ipynb) extrracted by the first LLM 1. 
As a first step the evaluation should be done only for the direct CI impacts - CI type, damage and geolocation

Issue:
* What is needed an approach that recognizes when an direct impact case is not detected by the model
Idea: 
* Split the original texts passed to the model on the exact chunks as again
* Then chunkwise check if the CI impacts from the validation set correspond in number and their textual similarity to the CI impacts infered by the LLM 1 and Entity Linking 

## Semantic Textual Similarity (STS)

Calculating the STS for both model configurations (chain of prompts, orchestration of models)
The outputs are cosine similarity scores for similar model outputs per chunk. They are ranked by score for each model, restricted to the top 20 results.  


In [247]:
import os
import sys
from pathlib import Path
import io
import gc
import time
import warnings
import subprocess
import importlib

from unidecode import unidecode
import langdetect
from fuzzywuzzy import fuzz
import torch
from huggingface_hub import login
import numpy as np
import spacy
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from matplotlib import pyplot as plt


sys.path.append('../')
from src.settings import settings as s
import src.document_cleaning as dc
import src.translation_model as tm
from src.utils import cosine_similarity, vector_calculation

# login to HF
# NOTE raises exception when env.variable does not exist (compared to os.envrion.get and its shortcut os.getenv)
os.getenv("HUGGINGFACE_TOKEN")

#  automatic linebreaks and multi-line cells.
pd.set_option("display.colheader_justify", "left")
pd.set_option('display.max_colwidth', 5000)


### Direct CI impacts: LLM 1 vs domain-expertise 

In [2]:
#  Suppress future warnings from PyTorch
warnings.filterwarnings("ignore", category=FutureWarning)


#  Define data dir where tags.csv and domain-expertise derived tag lists are found 
VALID_DATA_FILENAME = s.VALID_DATA_FILENAME
PATH_VALID_DATA = s.PATH_VALID_DATA
PATH_EVAL_RESULT = s.PATH_EVAL_RESULT
LLM_DATA_FILEPATH = Path(s.PATH_LLM_DATA / s.LLM_DATA_FILENAME)
SIMILARITY_LLM_FILENAME = s.SIMILARITY_LLM_FILENAME

df_valid_org = pd.read_csv(
    PATH_VALID_DATA / VALID_DATA_FILENAME,
    usecols=["publication_id", "ci1_type", "ci1_damage", "ci1_location", "sentence_reference"]
)
print(len(df_valid_org))
## pre-process: 
# remove undone entries
df_valid_org = df_valid_org[~df_valid_org.astype(str).apply(lambda x: x.str.contains("xx")).any(axis=1)]
# remove further location info (e.g. that entry is a town, Landkreis, Bavaria etc.)
df_valid_org["ci1_location"] = df_valid_org["ci1_location"].replace(r"\s*\(.*\)", "", regex=True).str.strip()
df_valid_org = df_valid_org.dropna(subset=["publication_id"], how="all") # drop rows where citation info is missing
print(len(df_valid_org))


## prediction data
df_pred = pd.read_csv(
    LLM_DATA_FILEPATH,
    #usecols=["citation_id", "chunk_id", "infrastructure_type", "damage", "location", "chunk_text"]
)

## citation alignment
# print(df_pred["citation_id"])
# df_pred["citation_id"] = df_pred["citation_id"].map(dc.extract_citation_info) # FIXME as used with new funct returning author, year, title
# df_pred["citation_id"] = df_pred["citation_id"].apply(dc.extract_citation_info)
# print(df_pred["citation_id"])


141
134


#### AS FUNC: Evaluate on same documents that were passed to LLM



In [3]:
#  TODO make as global var

PARSED_TEXT_DIR = Path(s.PATH_DATA / "parsed_documents/")

docs_list_sample = [
        Path(PARSED_TEXT_DIR, "Karakatsani 2023 - Greece economy briefing The economic impact of the recent devastating floods in Greece_cleaned.md"),
        Path(PARSED_TEXT_DIR, "Lloyd's List 2024 - Port of Valencia reopens after devastating floods_cleaned.md"),
        Path(PARSED_TEXT_DIR, "Containerlift 2024 - Valencia Port Resumes Operations Following Devastating Flooding in Spain - Containerlift.co.uk - Transport_Lifting_Shipping_cleaned.md"), 
        Path(PARSED_TEXT_DIR, "ABC 2024 - Traffic jams and flight delays due to heavy rain and lightning storm in Malaga_cleaned.md"),
        Path(PARSED_TEXT_DIR, "Koks 2022 - Brief communication_cleaned.md"),
        Path(PARSED_TEXT_DIR, "European Investment Bank 2025 - Spain_ EIB lends €50 million to Iberdrola to rebuild and climate-proof flood-hit power infrastructure in Valencia_cleaned.md"),
        Path(PARSED_TEXT_DIR, "Wilson 2024 - Flash floods in Spain sweep away cars, disrupt trains and leave several missing _ AP News_cleaned.md"),     
        Path(PARSED_TEXT_DIR, "Wildhagen 2013 - Hochwasser_ Wie die Flut Unternehmen lahmlegt_cleaned.md"),

        Path(PARSED_TEXT_DIR, "EFE 2024 - The DANA storm, live_ The death toll rises to 158_cleaned.md"),
        Path(PARSED_TEXT_DIR, "Ferlita 2023 - Incendi in Sicilia, ecco cosa accade_cleaned.md"),
        Path(PARSED_TEXT_DIR, "Gilbody Dickerson 2024 - Spain floods_ At least 95 people killed including British man near Malaga _ World News _ Sky News_cleaned.md"),

]



: 

In [ ]:
citation_list = []

for i in docs_list_sample:
    a, y, t = dc.extract_citation_info(i.name)
    citation_list.append(a + y)

df_valid = df_valid_org[df_valid_org["publication_id"].isin(citation_list)]


(80, 5)

In [5]:
df_pred[~df_pred["ci_entity"].isna()]


,citation_id,chunk_id,infrastructure_type,damage,location,ci_entity,geo_entity,case_type,chunk_text
0,Karakatsani 2023,0,road,severely damaged,Thessaly plain,the damages,Greece,NaN,"The economic impact of the recent devastating floods in Greece The economic impact of the recent devastating floods in Greece The briefing presents the economic impact of the damages caused by the storm “Daniel”. The floods caused by the heavy rainfall, especially in Thessaly plain -accounting for approximately 15% of Greece’s agricultural land- resulted to a massive destruction in agriculture, infrastructure and residences, which is expected to negatively affect the Greek economy in short and mid-term. Fears for food shortages and increase of product prices have been raised. In addition, increase of fiscal costs, imports and unemployment may also affect the economy in the upcoming years. The extent of the impact to the economy is not yet know. Before the country recovered from the wildfires of August and the consequent huge forest disaster, a weather stormy phenomenon named Daniel hit the country. Specifically, in the beginning of September, rainfall of unprecedented intensity fell mainly in central Greece and especially in Thessaly plain, causing floods throughout the territory. Thousands of acres of crops and livestock farms were destroyed. Properties and houses were lost under ton of waters. National roads were closed, bridges collapsed, and some parts of the railway network were highly damaged dividing the country in two. Evidently, the massive destruction of agricultural"
1,Karakatsani 2023,0,bridge,collapsed,Thessaly plain,National roads,plain,NaN,"The economic impact of the recent devastating floods in Greece The economic impact of the recent devastating floods in Greece The briefing presents the economic impact of the damages caused by the storm “Daniel”. The floods caused by the heavy rainfall, especially in Thessaly plain -accounting for approximately 15% of Greece’s agricultural land- resulted to a massive destruction in agriculture, infrastructure and residences, which is expected to negatively affect the Greek economy in short and mid-term. Fears for food shortages and increase of product prices have been raised. In addition, increase of fiscal costs, imports and unemployment may also affect the economy in the upcoming years. The extent of the impact to the economy is not yet know. Before the country recovered from the wildfires of August and the consequent huge forest disaster, a weather stormy phenomenon named Daniel hit the country. Specifically, in the beginning of September, rainfall of unprecedented intensity fell mainly in central Greece and especially in Thessaly plain, causing floods throughout the territory. Thousands of acres of crops and livestock farms were destroyed. Properties and houses were lost under ton of waters. National roads were closed, bridges collapsed, and some parts of the railway network were highly damaged dividing the country in two. Evidently, the massive destruction of agricultural"
2,Karakatsani 2023,0,railway,heavily damaged,Thessaly plain,bridges,plain,NaN,"The economic impact of the recent devastating floods in Greece The economic impact of the recent devastating floods in Greece The briefing presents the economic impact of the damages caused by the storm “Daniel”. The floods caused by the heavy rainfall, especially in Thessaly plain -accounting for approximately 15% of Greece’s agricultural land- resulted to a massive destruction in agriculture, infrastructure and residences, which is expected to negatively affect the Greek economy in short and mid-term. Fears for food shortages and increase of product prices have been raised. In addition, increase of fiscal costs, imports and unemployment may also affect the economy in the upcoming years. The extent of the impact to the economy is not yet know. Before the country recovered from the wildfires of August and the consequent huge forest disaster, a we

In [6]:
df_pred.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1299 entries, 0 to 1298
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   citation_id          1299 non-null   object 
 1   chunk_id             1299 non-null   int64  
 2   infrastructure_type  1299 non-null   object 
 3   damage               1299 non-null   object 
 4   location             1299 non-null   object 
 5   ci_entity            63 non-null     object 
 6   geo_entity           63 non-null     object 
 7   case_type            0 non-null      float64
 8   chunk_text           1299 non-null   object 
dtypes: float64(1), int64(1), object(7)
memory usage: 91.5+ KB


In [7]:
df_pred.loc[df_pred["citation_id"]== "Krausmann 2014"] # Krausmann -> large hallucinations when chunk-text is title or contact info (i.e when not about CI /impacts)

,citation_id,chunk_id,infrastructure_type,damage,location,ci_entity,geo_entity,case_type,chunk_text


### drop dublicated cases which differ only in Tier 2 or Tier 3 impacts

> e.g. valid ABC 2024: - identical c1_type, ci1_damage, ci1_loc (but diff. ci2_damages -which are not used in this eval) 


In [8]:

print(f"Dropping {df_valid.duplicated().sum()} duplicates in valid data")
df_valid = df_valid.drop_duplicates()

print(f"Dropping {df_pred.duplicated().sum()} duplicates in pred data")
df_pred = df_pred.drop_duplicates()


Dropping 1 duplicates in valid data
Dropping 57 duplicates in pred data


In [9]:
 # df_pred.sort_values(["citation_id", "chunk_id", "infrastructure_type"]).loc[df_pred.duplicated(keep=False, subset=["citation_id", "chunk_id", "infrastructure_type", "damage", "location", "ci_entity", "geo_entity", "case_type", "chunk_text"])][50:]

### add unique identifiers
helps in calculating FPs and FNs

In [10]:
df_pred["id_pred"] = df_pred.reset_index().index
df_valid["id_valid"] = df_valid.reset_index().index

#### As FUNC. postprocess -make CI gsubgroups

### Improve similarity calculation
As all similarity measures - no matter which embedding model or kind of cosine similarity measure were not sufficient eg. port ~ power to similar to port~harbor

Thus, it might be better to first group ci impacts into subgroups e.g .based on HARCI-EU categories,as some kind of postprocessing step before applying the similarity measurements



In [11]:
ci_patterns = pd.read_json("./ner_patterns.jsonl/patterns", lines=True)

In [12]:
import re

# load regular expressions and subgroups from NER patterns as dict
# for general cases and all special cases with "LOWER"-pattern
regexes_1 = [
        {ci_patterns["pattern"][i][0]["TEXT"]["REGEX"] : ci_patterns["subgroup_name"][i]}
          for i in range(len(ci_patterns)) 
            if len(ci_patterns["pattern"][i])==1 
]
regexes_2 = [
    {ci_patterns["pattern"][i][0]["TEXT"]["REGEX"] + " " + ci_patterns["pattern"][i][1]["LOWER"] : ci_patterns["subgroup_name"][i] }
      for i in range(len(ci_patterns))
        if len(ci_patterns["pattern"][i])==2
]
regexes = regexes_1 + regexes_2


In [13]:
for i, r in enumerate(regexes):

    # get regex pattern for CI type (key) and its subgroup (value)
    # NOTE, nice shortcut: get key containing regex by unpacking each dict into list, then get key
    pattern = [*r][0]
    subgroup = r[pattern]

    # assign subgroups to CI records, na=False to remove all records which not match patterns
    mask = df_pred["infrastructure_type"].str.contains(pattern, regex=True, na=False)
    df_pred.loc[mask, "infrastructure_group"] = subgroup

    mask = df_valid["ci1_type"].str.contains(pattern, regex=True, na=False)
    df_valid.loc[mask, "ci1_group"] = subgroup
    

print(df_pred.infrastructure_group.isna().sum())  # mostly cases which are not CI (theater, stadion..)
print(df_pred.infrastructure_group.value_counts()) # four most common subgroups seems to be correct
# df_pred.infrastructure_group.unique()



321
infrastructure_group
ports                           149
road_others                     120
rail                             95
healthcare_others                92
it_telecommunication             91
airports                         79
bridges                          53
education_kita                   43
transport_others                 41
education_others                 36
aviation                         27
healthcare_hospitals_clinics     19
water_supply                     17
education_school                 14
motorways                        14
wastewater                        7
electricity_others                5
water_others                      4
waste_others                      4
power_plants                      3
waterprotection                   3
water_distribution                1
drinking_water                    1
railway_station                   1
metro                             1
electricity_distribution          1
Name: count, dtype: int64


In [14]:
print(df_valid.ci1_group.isna().sum())
print(df_valid.ci1_group.value_counts())
# df_pred.infrastructure_group.unique()

12
ci1_group
road_others                     20
rail                             8
electricity_others               6
bridges                          4
healthcare_hospitals_clinics     4
airports                         3
wastewater                       3
motorways                        3
ports                            2
gas_distribution                 2
healthcare_others                2
water_others                     2
it_telecommunication             2
drinking_water                   2
waste_others                     1
water_supply                     1
education_kita                   1
education_school                 1
Name: count, dtype: int64


In [15]:
print("Removing all records which have another or erroneous CI entry")

df_pred = df_pred[~df_pred.infrastructure_group.isna()]
df_valid = df_valid[~df_valid.ci1_group.isna()]


Removing all records which have another or erroneous CI entry


#### Load spaCy language model


In [16]:
## load english model with word vectors included

print(f"Try loading spaCy language model ({s.SPACY_MODEL}) for remote instance (e.g., cluster)")
try: 
    nlp = spacy.load(s.SPACY_MODEL)
except (OSError, ValueError):
    print(f"spaCy language model '{s.SPACY_MODEL}' not found. Downloading ...")
    subprocess.check_call(["uv", "pip", "install", "spacy-transformers"])
    subprocess.check_call(["uv", "run", "python", "-m", "spacy", "download", s.SPACY_MODEL])
    nlp = spacy.load(s.SPACY_MODEL)

# NOTE: en_core_web_lg can only return word vectors, while en_core_web_trf return contextual vectors (as transformer-based)

# !uv run python -m spacy download en_core_web_lg
# nlp = spacy.load("en_core_web_lg")


Try loading spaCy language model (en_core_web_trf) for remote instance (e.g., cluster)


In [17]:
# ## unify citation column

# # get corresponding document from df_vald
# citation_pattern = r"(.*?)(\d{4})(.*)" # split at first occurrence of year
# # df_pred["publication_id"] = df_pred["citation"].map(dc.extract_citation_info)
# df_pred["citation"].map(dc.extract_citation_info)
# # df_pred = df_pred.rename({"citation": "publiation_id"})
# # df_pred.drop("citation", inplace=True)
# df_pred
# # try:
# #     authors, year, _ = re.findall(citation_pattern, filename)[0]
# #     citation = f"{authors} {year}"


#### Select records which have text references

In [18]:
df_valid_org.info()

<class 'pandas.core.frame.DataFrame'>
Index: 134 entries, 0 to 140
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   publication_id      134 non-null    object
 1   sentence_reference  126 non-null    object
 2   ci1_type            109 non-null    object
 3   ci1_damage          96 non-null     object
 4   ci1_location        88 non-null     object
dtypes: object(5)
memory usage: 6.3+ KB


In [19]:

print(len(df_pred), len(df_valid))
df_pred = df_pred[~df_pred["chunk_text"].isna()].reset_index(drop=True)
df_valid = df_valid[~df_valid["sentence_reference"].isna()].reset_index(drop=True)
print(len(df_pred), len(df_valid))


921 67
921 67


#### Translation of validation sentences

In [20]:

for entry in df_valid.itertuples():
    
    src_language = langdetect.detect(str(entry.sentence_reference))
    
    if src_language != "en":
        supported_languages = ["fr", "de", "es", "it", "itc", "nl"]
        if src_language not in supported_languages:
            print(f"Unsupported source language: {src_language}. Continue with original version of the sentence in validation set ")
            continue 

        print(f"\n ######## -------- Translating {entry.publication_id}: {src_language} --> en -------- ######## \n")

        # # clean up before applying translator
        # gc.collect()
        # torch.cuda.empty_cache()  # mainly after training needed, small effect when LLM applied only for inference
        # torch.no_grad()
        
        # overwrite original sentence(s) with translated versions
        translated_sentence = tm.translate_2_english(src_language, str(entry.sentence_reference))
        df_valid.loc[df_valid.index[df_valid["sentence_reference"] == entry.sentence_reference], "sentence_reference"] = translated_sentence



 ######## -------- Translating Ferlita 2023: it --> en -------- ######## 

/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval/notebooks/huggingface_mirror/hub
Using device: cuda
Using locally saved model from /home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval/notebooks/huggingface_mirror/hub
Input document is a string (not DoclingObject). Wrapping it in a list for processing.
Continue with translation of text string


#### Postprocess (text cleaning)


In [21]:
df_valid.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67 entries, 0 to 66
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   publication_id      67 non-null     object
 1   sentence_reference  67 non-null     object
 2   ci1_type            67 non-null     object
 3   ci1_damage          61 non-null     object
 4   ci1_location        54 non-null     object
 5   id_valid            67 non-null     int64 
 6   ci1_group           67 non-null     object
dtypes: int64(1), object(6)
memory usage: 3.8+ KB


In [22]:
# unicode to ascii representation

for col in ["infrastructure_group", "infrastructure_type", "damage", "ci_entity", "geo_entity"]:
    df_pred[col] = df_pred[col].apply(lambda x: unidecode(x) if isinstance(x, str) else x) # handle potential np.nan


for col in ["ci1_group", "ci1_type", "ci1_damage", "ci1_location"]:
    df_valid[col] = df_valid[col].apply(lambda x: unidecode(x) if isinstance(x, str) else x) # handle potential np.nan


#### Merge prediction entries with potential validation entries (nth:1 pairs)

In [23]:
print("Match chunk text of each prediction entry with related validation entries (nth:1 pairs)") # getting nth:1 pairs

df_pred_valid_all = pd.DataFrame()
threshold = 75

# find for each prediction entry all validation entries for respective chunk 
# these validation entries are candidates from which the most similar one to the pred. entry is taken to calc. model performance 
# including also entries where pred_info or valid_info is missing (e.g FNs, FPs)
for _, pred_entry in df_pred.iterrows():
    for _, valid_entry in df_valid.iterrows():  # all validation entries of all docs

        if valid_entry.sentence_reference is np.nan:
            continue

        # Calculate match score by accounting for partial string matches. 
        # In detail, it calculates the similarity ratio using the shortest string (length n, here: "sentence_reference") against all n-length substrings of the larger string and returns the highest score 
        score = fuzz.partial_ratio(valid_entry['sentence_reference'], pred_entry['chunk_text'])

        if score >= threshold:
            entry_pred_valid = {
                "citation_id": pred_entry["citation_id"],
                "ci_pred": pred_entry["infrastructure_type"],
                "ci_group_pred": pred_entry["infrastructure_group"],
                "damage_pred": pred_entry["damage"],
                "location_pred": pred_entry["location"],
                "chunk_id_pred": pred_entry["chunk_id"],
                "chunk_text_pred": pred_entry["chunk_text"],
                "ci_valid": valid_entry["ci1_type"],
                "ci_group_valid": valid_entry["ci1_group"],
                "damage_valid": valid_entry["ci1_damage"],
                "location_valid": valid_entry["ci1_location"],
                "sentence_text_valid": valid_entry["sentence_reference"],
                "text_similarity": score,
                "id_pred": pred_entry["id_pred"],
                "id_valid": valid_entry["id_valid"]
            }
            df_pred_valid_all = pd.concat([df_pred_valid_all, pd.DataFrame([entry_pred_valid])], ignore_index=True)  # n:1 relationship DF
        

# 85 threshold - 778 entries
# 75 threshold - 778 entries
# 75 threshold + CIsubgrou - 513 entries




Match chunk text of each prediction entry with related validation entries (nth:1 pairs)


In [32]:
df_pred_valid_all.info() # 143 -190 entries

## --> FPs are more common compared to FNs, especially for predicting locations, 
# as it is easier to get a prep-valid match when pred.info is actually missing due to larger chunk-text (pred set) compared to sentence-text (valid set)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 309 entries, 0 to 308
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   citation_id          309 non-null    object
 1   ci_pred              309 non-null    object
 2   ci_group_pred        309 non-null    object
 3   damage_pred          309 non-null    object
 4   location_pred        309 non-null    object
 5   chunk_id_pred        309 non-null    int64 
 6   chunk_text_pred      309 non-null    object
 7   ci_valid             309 non-null    object
 8   ci_group_valid       309 non-null    object
 9   damage_valid         273 non-null    object
 10  location_valid       274 non-null    object
 11  sentence_text_valid  309 non-null    object
 12  text_similarity      309 non-null    int64 
 13  id_pred              309 non-null    int64 
 14  id_valid             309 non-null    int64 
dtypes: int64(4), object(11)
memory usage: 36.3+ KB


In [25]:
## entries with lowest similarity
df_pred_valid_all.text_similarity.describe()
# df_pred_valid_all.iloc[df_pred_valid_all.text_similarity.sort_values(ascending=True).index] [["sentence_text_valid", "chunk_text_pred","text_similarity"]]

count    309.000000
mean      99.346278
std        2.165212
min       87.000000
25%      100.000000
50%      100.000000
75%      100.000000
max      100.000000
Name: text_similarity, dtype: float64

#### Calc similarities for all cases where text info in pred and valid set exists 


In [26]:

# ## TODO test to prevent CUDA-OOM when reused
# ## Source: https://spacy.io/usage/embeddings-transformers
# from thinc.api import set_gpu_allocator, require_gpu

# # Use the GPU, with memory allocations directed via PyTorch.
# # This prevents out-of-memory errors that would otherwise occur from competing
# # memory pools.
# set_gpu_allocator("pytorch")
# require_gpu(0)

In [27]:
columns_valid = ["ci_group_valid", "damage_valid", "location_valid"]
columns_pred = ["ci_group_pred", "damage_pred", "location_pred"]



#  Set similarity threshold (self-defined) when CI case is valid or not FN/FP
similarity_threshold = 0.7


print(" --- For each unique valid case (unique combi: [ci_valid, damage_valid, location_valid, sentence_text]) calculate similarity ---")
print("Using 100% match for Ci types based on its subgroups")
print("Using as cosine similarity threshold for damages and location:", similarity_threshold)

## AIM of evaluation loop below: 
# remove all cases in df_pred_valid_all where pred_entities were wrongly assigned to a valid_entity
## ie keep only pre-valid pairs with highest similarity per unique valid case

# for each impact type (ci, damage, location) assess LLM performance
for column_valid, column_pred in zip(columns_valid, columns_pred):

    df_smltry_all = pd.DataFrame()

    print(f"\n --------- Calculate similarities for entries in column pair: {column_pred} - {column_valid} ------------")


    ## calculate similarities
    for _, entry in df_pred_valid_all.iterrows():

        ## calc similarity when both pred_info and valid_info exist (ie. not NaN)
        if entry[column_pred] and entry[column_valid] is not np.nan:
            pred_impact = entry[column_pred]
            valid_impact = entry[column_valid]

            if column_pred == "ci_group_pred": # for CI group, only partial ratio similarity is calculated as it is more important to get the correct group than the exact match (e.g. "port infrastructure" <-> "port")
                
                # FIXME plot here cos as partial sim. jsut to keep info how simi changes between cos ~ subgrouping)
                embedded_list = vector_calculation(pred_impact, valid_impact)
                similarity_score_cos = cosine_similarity(embedded_list[0], embedded_list[1])
                similarity_score_pr = np.nan
                # calc similarity based on subgroups (100% match)
                if pred_impact == valid_impact:
                    similarity_score = 1
                else:
                    similarity_score = 0

            else:
                ## Cosine similarity calc.
                # contextual vectors (transformer-based)
                embedded_list = vector_calculation(pred_impact, valid_impact)
                # calculate cosine similarity for each pred-valid pair
                similarity_score = np.nan
                similarity_score_cos = cosine_similarity(embedded_list[0], embedded_list[1])  # 0-1 value, the higher the more similar

                ## Partial ratio similarity calc. (especially for locations and CI-type  "port infrastructure" <-> "port")
                similarity_score_pr = fuzz.partial_ratio(pred_impact, valid_impact)  


            # store results
            dict_pair = {
                "impact_pred": pred_impact, # grouped version for CI !!
                "impact_valid": valid_impact, # grouped version for CI !! TODO give also non-grouped CI types
                "impact_sim_identical": np.round(similarity_score, 2),
                "impact_sim_cos": np.round(similarity_score_cos, 2),
                "impact_sim_pr": similarity_score_pr,
                "tp_tn_fp_fn": "tp_tn",
                "citation": entry.citation_id,
                "chunk_text_pred": entry.chunk_text_pred,
                "sentence_text_valid": entry.sentence_text_valid,
                "id_pred": entry.id_pred,
                "id_valid": entry.id_valid
                }
            df_smltry_all = pd.concat([df_smltry_all, pd.DataFrame([dict_pair])], ignore_index=True)


    # when no similarity could be calculated
    if column_pred == "ci_group_pred":
        entries_with_no_similarity = df_smltry_all.loc[df_smltry_all["impact_sim_identical"].isna()]
        print(f" --- Pred-valid pairs where no identical similarity score could be calculated: {len(entries_with_no_similarity)} ----")
        print(entries_with_no_similarity[["impact_valid", "impact_pred", "impact_sim_identical", "impact_sim_cos", "impact_sim_pr", "citation"]])

        # handle when multiple rows have highest similarity score
        # mask of rows with highest similarity score for each set of preds with unique valid case (droplevel(0) remove multiindex)
        mask = df_smltry_all.groupby("id_valid").apply(lambda x: x==x["impact_sim_identical"].max()).droplevel(0)
        df_smltry_selmax = df_smltry_all.where(mask.impact_sim_identical==mask.impact_sim_identical.max()).dropna(how="all") # drop cases which have not highest similarity score
        df_smltry_selmax.reset_index(drop=True, inplace=True)
        # old: returns only first row with highest sim. score
        # df_smltry_selmax = df_smltry_all.groupby("id_valid").apply(lambda x: x.loc[x["impact_sim_identical"].idxmax()])

    else:
        entries_with_no_similarity = df_smltry_all.loc[df_smltry_all["impact_sim_cos"].isna()]
        print(f" --- Pred-valid pairs where no cos. similarity score could be calculated: {len(entries_with_no_similarity)} ----")
        print(entries_with_no_similarity[["impact_valid", "impact_pred", "impact_sim_identical", "impact_sim_cos", "impact_sim_pr", "citation"]])

        # handle when multiple rows have highest similarity score
        # mask of rows with highest similarity score for each set of preds with unique valid case (droplevel(0) remove multiindex)
        mask = df_smltry_all.groupby("id_valid").apply(lambda x: x==x["impact_sim_identical"].max()).droplevel(0)
        df_smltry_selmax = df_smltry_all.where(mask.impact_sim_identical==mask.impact_sim_identical.max()).dropna(how="all") # drop cases which have not highest similarity score
        df_smltry_selmax.reset_index(drop=True, inplace=True)
        # old: returns only first row with highest sim. score
        # df_smltry_selmax = df_smltry_all.groupby("id_valid").apply(lambda x: x.loc[x["impact_sim_identical"].idxmax()])

    print("for each unique valid_cols keep only pred-valid pairs of highest similarity")
    

    print("calculate performance measures:")

    # Ci type grouped
    if column_pred == "ci_group_pred":
        
        # TPs 
        tps = df_smltry_selmax.loc[df_smltry_selmax["impact_sim_identical"] == 1]

        # FPs
        # get records where model predicted presence of impacts but they actually does not exist
        # here as definition, that when simi=0 (or below threshold) then model predicted a falsealarm 
        # ???? 
        fps = df_smltry_selmax.loc[df_smltry_selmax["impact_sim_identical"] == 0]
        # TODO
        # add also as Fps were model_pred case exist but no fitting_vlaid case could be found (during df_valid_pred pair generation in loop at begin of NB)

        df_valid_pred_same_docs = df_valid[df_valid["publication_id"].isin(df_pred["citation_id"])]
        print(f"Doing evaluation based on {df_valid_pred_same_docs.publication_id.unique().__len__()} documents existing in both (valid.+pred. set)")

        # FNs
        ## missed docs
        df_valid_pred_missed_docs = df_valid[df_valid["publication_id"].isin(df_pred["citation_id"]) == False]
        ## missed entries
        # extracts all duplicates (except first occurrence eg. id_Pred==537 occurs in df_smltry_selmax_p three times (1st case: TP or FP, 2nd and 3rd are FNs)
        df_valid_cases_missed_by_model = df_smltry_selmax[df_smltry_selmax.duplicated(subset="id_pred", keep="first")]
        ## OLD APPROACH: CI cases in valid set (for docs existing in both sets) - number of corectly predicted CI cases (Tps)
        ## no_valid_cases_missed_by_model  = len(df_valid_pred_same_docs["ci1_group"])  - len(df_smltry_selmax[df_smltry_selmax["impact_sim_identical"]==1])

        ## TODO FIXME not sure if approach for df_valid_cases_missed_by_model based on df_smltry_selmax is correct
        ##            as df_smltry_selmax contains only the cases of highest similarity for each case in df_valid (ie unique id_valid)
        ##            can i then calc the number of missed cases by 
        
        fns = pd.concat([df_valid_pred_missed_docs, df_valid_cases_missed_by_model], ignore_index=True, axis=0)

        # FIXME WORKAROUND for FP and FN calculation, but not extract respective cases (only numbers of FPs and FNs)
        if column_pred == "ci_group_pred":
            fps_len = len(df_pred["ci_group"]) - tps.shape[0]
            fns_len = len(df_valid["ci1_group"]) - tps.shape[0]
        elif column_pred == "location_pred":
            fps_len = len(df_pred["location"]) - tps.shape[0]
            fns_len = len(df_valid["ci1_location"]) - tps.shape[0]
        print("tps", len(tps), " fps:", fps_len, " fns:", fns_len)


    # damage, location
    else:

        # TPs 
        tps = df_smltry_selmax.loc[df_smltry_selmax["impact_sim_identical"] >= similarity_threshold]
        
        # FPs - model predicts condition wrongly (ie. predict condition when it is actually absent)
        # get all valid. documents which were also used for LLM inference
        df_valid_pred_same_docs = df_valid[df_valid["publication_id"].isin(df_pred["citation_id"])]
        print(f"Doing evaluation based on {df_valid_pred_same_docs.publication_id.unique().__len__()} documents existing in both (valid.+pred. set)")
        # get records where model predicted presence of impacts but they actually does not exist
        fps = df_smltry_selmax.loc[df_smltry_selmax["impact_sim_identical"] < similarity_threshold]
        # here as definition, that when simi=0 (or below threshold) then model predicted presences as false alarm
        # TODO
        # add also as Fps were model_pred case exist but no fitting_valid case could be found (during df_valid_pred pair generation in loop at begin of NB)
        # df_pred selction needed

        # FNs - CI impact cases not detected by model 
        # NOTE: maybe FNs number is biased as wrong matches more likely as chunk-text (pred set) is longer than sentence text (valid set)
        # WRONG? get all entries from df_valid_pred_same_docs where corresponding pred_record (in FPs) is missing

        # get all documents in valid_set which does not occur in pred_set or where similarity is too low
        ## missed docs
        df_valid_pred_missed_docs = df_valid[df_valid["publication_id"].isin(df_pred["citation_id"]) == False]
        print("Number of documents where model did not extract anything", df_valid_pred_missed_docs.shape)
        ## missed entries
        # extracts all duplicates (except first occurrence eg. id_Pred==537 occurs in df_smltry_selmax_p three times (1st case: TP or FP, 2nd and 3rd are FNs)
        df_valid_cases_missed_by_model = df_smltry_selmax[df_smltry_selmax.duplicated(subset="id_pred", keep="first")]
        ## OLD APPROACH: CI cases in valid set (for docs existing in both sets) - number of correctly predicted CI cases (Tps)
        # no_valid_cases_missed_by_model  = len(df_valid_pred_same_docs["ci1_damage"])  - len(df_smltry_selmax["impact_valid"])
        # no_valid_cases_missed_by_model  = len(df_valid_pred_same_docs["ci1_location"])  - len(df_smltry_selmax["impact_valid"])
        fns = pd.concat([df_valid_pred_missed_docs, df_valid_cases_missed_by_model], ignore_index=True, axis=0)
        
        # FIXME WORKAROUND for FP and FN calculation, but not extract respective cases (only numbers of FPs and FNs)
        if column_pred == "damage_pred":
            fps_len = len(df_pred["damage"]) - tps.shape[0]
            fns_len = len(df_valid["ci1_damage"]) - tps.shape[0]
        elif column_pred == "location_pred":
            fps_len = len(df_pred["location"]) - tps.shape[0]
            fns_len = len(df_valid["ci1_location"]) - tps.shape[0]
        print("tps", len(tps), " fps:", fps_len, " fns:", fns_len)

    # performance scores
    recall_score = len(tps) / (fps_len + fns_len) 
    precision_score = len(tps) / (len(tps) + fps_len)
    f1_score = 2 * (precision_score * recall_score) / (precision_score + recall_score)


    print(f" ---------- Evaluation statistics: {column_pred}-----------")
    print(f"Recall: {recall_score}, Precision: {precision_score}, F1-score: {f1_score}")


    SIMILARITY_FILENAME = f"{column_pred.replace('_pred', '')}_{SIMILARITY_LLM_FILENAME}"
    SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)


    print("Saving evaluation statistics, distribution plots, and scores to ", SIMILARITY_FILEPATH.stem, "[.parquet, _stats.json]")
    
    with open(SIMILARITY_FILEPATH, "w") as f:

        # results similarity df (with highest similarity per each valid_identifier) [csv, parquet]
        df_smltry_selmax.to_csv(SIMILARITY_FILEPATH.with_suffix('.csv'), index=False)
        df_smltry_selmax_pyarrow = pa.Table.from_pandas(df_smltry_selmax.astype( dtype="string[pyarrow]"))
        pq.write_table(df_smltry_selmax_pyarrow, SIMILARITY_FILEPATH.with_suffix(".parquet"))

        # results similarity all (all cases where valid_records was used multiple times to calc similairty to preds
        df_smltry_all.to_csv(SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_allmatches.csv", index=False) 

        # summary statistics
        df_smltry_selmax_stats = pd.DataFrame(
            [{"recall": np.round(recall_score, 2), "precision": np.round(precision_score, 2), "f1_score": np.round(f1_score, 2)}]
        )
        f = SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_stats.json"  
        df_smltry_selmax_stats.to_json(f, indent=4)
        
        # distribution plots
        if column_pred == "ci_group_pred":
            df_smltry_selmax["impact_sim_identical"].hist(bins=100)
            plt.savefig(SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_hist.png")
            plt.close()

            df_smltry_all["impact_sim_identical"].hist(bins=100)
            plt.savefig(SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_allmatches_hist.png")
            plt.close()        
        else:
            df_smltry_selmax["impact_sim_cos"].hist(bins=100)
            plt.savefig(SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_hist.png")
            plt.close()

            df_smltry_all["impact_sim_cos"].hist(bins=100)
            plt.savefig(SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_allmatches_hist.png")
            plt.close()        





 --- For each unique valid case (unique combi: [ci_valid, damage_valid, location_valid, sentence_text]) calculate similarity ---
Using 100% match for Ci types based on its subgroups
Using as cosine similarity threshold for damages and location: 0.7

 --------- Calculate similarities for entries in column pair: ci_group_pred - ci_group_valid ------------


 --- Pred-valid pairs where no identical similarity score could be calculated: 0 ----
Empty DataFrame
Columns: [impact_valid, impact_pred, impact_sim_identical, impact_sim_cos, impact_sim_pr, citation]
Index: []
for each unique valid_cols keep only pred-valid pairs of highest similarity
calculate performance measures:
Doing evaluation based on 9 documents existing in both (valid.+pred. set)
tps 45  fps: 7  fns: 22
 ---------- Evaluation statistics: ci_group_pred-----------
Recall: 0.6716417910447762, Precision: 0.8653846153846154, F1-score: 0.7563025210084034
Saving evaluation statistics, distribution plots, and scores to  ci_group_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]

 --------- Calculate similarities for entries in column pair: damage_pred - damage_valid ------------


KeyboardInterrupt: 

In [ ]:
# old
# Recall: 0.7377049180327869, Precision: 0.8653846153846154, F1-score: 0.7964601769911505

# fixed partly recall (FNs)
# Recall: 0.6716417910447762, Precision: 0.8653846153846154, F1-score: 0.7563025210084034

# new LLM extraction with fixed NERpatterns
# Recall: 0.6716417910447762, Precision: 0.8653846153846154, F1-score: 0.7563025210084034


#### FIXME: find out which cases model predicted existence, but not in valid DS - maybe due that valid DS is incomppete?

In [228]:
df_pred.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 921 entries, 0 to 920
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   citation_id           921 non-null    object 
 1   chunk_id              921 non-null    int64  
 2   infrastructure_type   921 non-null    object 
 3   damage                921 non-null    object 
 4   location              921 non-null    object 
 5   ci_entity             50 non-null     object 
 6   geo_entity            50 non-null     object 
 7   case_type             0 non-null      float64
 8   chunk_text            921 non-null    object 
 9   id_pred               921 non-null    int64  
 10  infrastructure_group  921 non-null    object 
dtypes: float64(1), int64(2), object(8)
memory usage: 79.3+ KB


In [ ]:
## fix FPs 

## get all pred cases which 
# rows in df_valid where sentence_reference appears as substring in at least one df_pred.chunk_text
chunk_texts = df_pred["chunk_text"].dropna().astype(str)

df_valid_2 = df_valid[
    df_valid["sentence_reference"].fillna("").astype(str).apply(
        lambda s: any(s and s in chunk for chunk in chunk_texts)
    )
]

df_valid_2 # .shape (28, 7)

,publication_id,sentence_reference,ci1_type,ci1_damage,ci1_location,id_valid,ci1_group
0,ABC 2024,Heavy rains and a severe thunderstorm brought by the DANA storm system have caused signiﬁcant trafﬁc jams and ﬂight delays at Malaga airport this Tuesday.,airport,affected,Malaga area,0,airports
1,Containerlift 2024,The Port of Valencia has reopened for operations after severe flooding temporarily halted activity across eastern Spain.,Port infrastructure,affected,Port of Valencia,1,ports
2,Containerlift 2024,"Nevertheless, the reopening of Valencia and Sagunto ports for maritime traffic marks a significant step forward for the",Port infrastructure,affected,Sagunto port,2,ports
3,EFE 2024,"The most serious disruptions are on roads in Valencia, with closures on several sections of the A-3, the A-7 and the AP-7, as well as on the roads that connect it sections of the A-3, the A-7 and the AP-7, as well as on the roads that connect it with Alicante and the N-3, N-322, N-330 and N-332 roads as they pass through the with Alicante and the N-3, N-322, N-330 and N-332 roads as they pass through the towns of Picassent, La Alcudia, Requena, Utiel, Buñol, Sueca, Algemesí, towns of Picassent, La Alcudia, Requena, Utiel, Buñol, Sueca, Algemesí, Guadassuar, Alzira or Chiva (Valencia), among other municipalities.",road,closures,Sections at A-3 in Valencia area,3,road_others
4,EFE 2024,"The most serious disruptions are on roads in Valencia, with closures on several sections of the A-3, the A-7 and the AP-7, as well as on the roads that connect it sections of the A-3, the A-7 and the AP-7, as well as on the roads that connect it with Alicante and the N-3, N-322, N-330 and N-332 roads as they pass through the with Alicante and the N-3, N-322, N-330 and N-332 roads as they pass through the towns of Picassent, La Alcudia, Requena, Utiel, Buñol, Sueca, Algemesí, towns of Picassent, La Alcudia, Requena, Utiel, Buñol, Sueca, Algemesí, Guadassuar, Alzira or Chiva (Valencia), among other municipalities.",road,closures,Sections at A-7 in Valencia area,4,road_others
5,EFE 2024,"The most serious disruptions are on roads in Valencia, with closures on several sections of the A-3, the A-7 and the AP-7, as well as on the roads that connect it sections of the A-3, the A-7 and the AP-7, as well as on the roads that connect it with Alicante and the N-3, N-322, N-330 and N-332 roads as they pass through the with Alicante and the N-3, N-322, N-330 and N-332 roads as they pass through the towns of Picassent, La Alcudia, Requena, Utiel, Buñol, Sueca, Algemesí, towns of Picassent, La Alcudia, Requena, Utiel, Buñol, Sueca, Algemesí, Guadassuar, Alzira or Chiva (Valencia), among other municipalities.",road,closures,Sections at AP-7 in Valencia area,5,road_others
6,EFE 2024,"The most serious disruptions are on roads in Valencia, with closures on several sections of the A-3, the A-7 and the AP-7, as well as on the roads that connect it sections of the A-3, the A-7 and the AP-7, as well as on the roads that connect it with Alicante and the N-3, N-322, N-330 and N-332 roads as they pass through the with Alicante and the N-3, N-322, N-330 and N-332 roads as they pass through the towns of Picassent, La Alcudia, Requena, Utiel, Buñol, Sueca, Algemesí, towns of Picassent, La Alcudia, Requena, Utiel, Buñol, Sueca, Algemesí, Guadassuar, Alzira or Chiva (Valencia), among other municipalities.",roads,closures,roads between Valencia and Alicante,6,road_others
7,EFE 2024,"The most serious disruptions are on roads in Valencia, with closures on several sections of the A-3, the A-7 and the AP-7, as well as on the roads that connect it sections of the A-3, the A-7 and the AP-7, as well as on the roads that connect it with Alicante and the N-3, N-322, N-330 and N-332 roads as they pass through the with Alicante and the N-3, N-322, N-330 and N-332 roads as they pass through the towns of Picassent, La Alcudia, Requena, Utiel, Buñol, Sueca, Algemesí, towns of Picassent, La Alcudia, Requena, Utie

In [ ]:
# cases where pred-case exist but no fitting valid case could be found based on sentence_reference
df_pred_not_in_valid = df_pred[df_pred['chunk_text'].str.contains('|'.join(df_valid["sentence_reference"]), regex=True)]
print(df_pred_not_in_valid.id_pred.value_counts())
df_pred_not_in_valid.head(10)

# TODO TODO
## documents where model found many CI cases as false-alarms:
# maybe i need to recheck those docs and make df_valid more complete
# print(df_pred_not_in_valid.groupby("citation_id").count())
# citation_id                                                            
# ABC 2024                        4
# Containerlift 2024             24
# European Investment Bank 2025  21
# Ferlita 2023                   22
# Koks 2022                      10


id_pred
200     1
201     1
202     1
204     1
205     1
       ..
1024    1
1025    1
1026    1
1027    1
1028    1
Name: count, Length: 81, dtype: int64
                               chunk_id  infrastructure_type  damage  \
citation_id                                                            
ABC 2024                        4         4                    4       
Containerlift 2024             24        24                   24       
European Investment Bank 2025  21        21                   21       
Ferlita 2023                   22        22                   22       
Koks 2022                      10        10                   10       

                               location  ci_entity  geo_entity  case_type  \
citation_id                                                                 
ABC 2024                        4        1          1           0           
Containerlift 2024             24        8          8           0           
European Investment Bank 2025  

/tmp/ipykernel_7335/1699970596.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_pred_not_in_valid = df_pred[df_pred['chunk_text'].str.contains('|'.join(df_valid["sentence_reference"]), regex=True)]


In [206]:
df_valid

,publication_id,sentence_reference,ci1_type,ci1_damage,ci1_location,id_valid,ci1_group
0,ABC 2024,Heavy rains and a severe thunderstorm brought by the DANA storm system have caused signiﬁcant trafﬁc jams and ﬂight delays at Malaga airport this Tuesday.,airport,affected,Malaga area,0,airports
1,Containerlift 2024,The Port of Valencia has reopened for operations after severe flooding temporarily halted activity across eastern Spain.,Port infrastructure,affected,Port of Valencia,1,ports
2,Containerlift 2024,"Nevertheless, the reopening of Valencia and Sagunto ports for maritime traffic marks a significant step forward for the",Port infrastructure,affected,Sagunto port,2,ports
3,EFE 2024,"The most serious disruptions are on roads in Valencia, with closures on several sections of the A-3, the A-7 and the AP-7, as well as on the roads that connect it sections of the A-3, the A-7 and the AP-7, as well as on the roads that connect it with Alicante and the N-3, N-322, N-330 and N-332 roads as they pass through the with Alicante and the N-3, N-322, N-330 and N-332 roads as they pass through the towns of Picassent, La Alcudia, Requena, Utiel, Buñol, Sueca, Algemesí, towns of Picassent, La Alcudia, Requena, Utiel, Buñol, Sueca, Algemesí, Guadassuar, Alzira or Chiva (Valencia), among other municipalities.",road,closures,Sections at A-3 in Valencia area,3,road_others
4,EFE 2024,"The most serious disruptions are on roads in Valencia, with closures on several sections of the A-3, the A-7 and the AP-7, as well as on the roads that connect it sections of the A-3, the A-7 and the AP-7, as well as on the roads that connect it with Alicante and the N-3, N-322, N-330 and N-332 roads as they pass through the with Alicante and the N-3, N-322, N-330 and N-332 roads as they pass through the towns of Picassent, La Alcudia, Requena, Utiel, Buñol, Sueca, Algemesí, towns of Picassent, La Alcudia, Requena, Utiel, Buñol, Sueca, Algemesí, Guadassuar, Alzira or Chiva (Valencia), among other municipalities.",road,closures,Sections at A-7 in Valencia area,4,road_others
...,...,...,...,...,...,...,...
62,Koks 2022,"Furthermore, in the region of Rhineland-Palatinate (Germany), 19 daycare centres and 17 schools suffered damage from the ﬂoods, affecting more than 8000 students (Staib, 2021).",daycare centers,damaged,Rhineland-Palantine,72,education_kita
63,Koks 2022,"Furthermore, in the region of Rhineland-Palatinate (Germany), 19 daycare centres and 17 schools suffered damage from the ﬂoods, affecting more than 8000 students (Staib, 2021).",schools,damaged,Rhineland-Palantine,73,education_school
64,Koks 2022,"In the Netherlands, one nursing home was ﬂooded, and one hospital was evacu- ated as a precautionary measure.",hospital,NaN,Netherlands,75,healthcare_hospitals_clinics
65,Wildhagen 2013,"Because bridges and roads in the brown Water masses sank, broke the logistics chains of many businesses and set, because parts and material were missing, the production was matt. Thus, Volkswagen stopped Trucks stopped at destroyed bridges, barges couldn't even stop at all",bridges,damaged,NaN,76,bridges


#### FIXME: FPs and FNs

In [ ]:
## TPs + FNs should be == len(df_valid.ci) == 67
        
# TPs 
tps = df_smltry_selmax.loc[df_smltry_selmax["impact_sim_identical"] == 1]

# FNs
df_valid_pred_missed_docs = df_valid[df_valid["publication_id"].isin(df_pred["citation_id"]) == False]
df_valid_cases_missed_by_model = df_smltry_selmax[df_smltry_selmax.duplicated(subset="id_pred", keep="first")]
fns = pd.concat([df_valid_pred_missed_docs, df_valid_cases_missed_by_model], ignore_index=True, axis=0)

print(tps.shape[0], fns.shape[0])
print(tps.shape[0] + fns.shape[0])

# --> 8 cases in FNs are too definitly too much --> fix FN calculation



## FPs should be == len(df_pred.ci) - TPs

## FPs
fps = df_smltry_selmax.loc[df_smltry_selmax["impact_sim_identical"] == 0]

print(len(df_pred.infrastructure_type), tps.shape[0], fps.shape[0])
print(len(df_pred.infrastructure_type) - tps.shape[0])




45 30
75
921 45 7
876


In [232]:
# TODO fix FPs
# get records where model predicted presence of impacts but they actually does not exist
# here as definition, that when simi=0 (or below threshold) then model predicted wrongly
df_smltry_not_sim = df_smltry_selmax.loc[df_smltry_selmax["impact_sim_identical"] == 0]

# TODO
# add also as Fps were model_pred case exist but no fitting_vlaid case could be found

# idea: 
# get all df_pred cases where chunk text not occurs in valid.sentece_text

df_pred_not_in_valid = df_pred[df_pred["chunk_text"].isin(df_valid["sentence_reference"])== False]
print(df_pred.shape, df_pred_not_in_valid.shape)
# df_pred_not_in_valid

(921, 11) (921, 11)


In [77]:
# df_valid__pred_no_thresh.id_pred.nunique()
df_pred_valid_no_thresh.id_pred.nunique()

921

In [ ]:
# df_valid__pred_no_thresh.info()
# df_pred_valid_no_thresh.info()  # 67 valid * 921 pred = 61707
# df_pred_valid_no_thresh.drop("chunk_text_pred", axis=1).sort_values("id_pred").iloc[0:100]
# df_pred_valid_no_thresh.groupby("id_pred").first().sort_values("text_similarity", ascending=False).iloc[0:100]
# df_valid__pred_no_thresh.groupby("id_valid").first().sort_values("text_similarity", ascending=False).iloc[0:100]
#df_valid__pred_no_thresh.sort_values("id_valid", ascending=False).sort_values("text_similarity", ascending=False).iloc[0:100]
#df_valid__pred_no_thresh.groupby("id_valid").first().sort_values("text_similarity", ascending=False).iloc[0:100]
df_valid__pred_no_thresh.groupby("id_valid").apply(lambda x: x.loc[x["text_similarity"].idxmax()])


In [ ]:
# fns

In [ ]:
## FNs 
df_smltry_selmax.loc[df_smltry_selmax.duplicated("id_pred")]
## --> ISSUE: this df (cases of highest sim) should NOT have duplicated cases of predictions -> maybe have to group based on id_pred and not id_valid

## try to fix issue
## --> currently i think this should group based on valid cases to measure were model predicted the same or missed info (ie FNs)
# df_smltry_selmax_p = df_smltry_selmax
# mask of rows with highest similarity score for each set of preds with unique valid case (droplevel(0) remove multiindex)
mask = df_smltry_all.loc[df_smltry_all.id_valid==37].groupby("id_valid").apply(lambda x: x==x["impact_sim_identical"].max()).droplevel(0)
df_smltry_selmax_p = df_smltry_all.where(mask.impact_sim_identical==mask.impact_sim_identical.max()).dropna(how="all") # drop cases which have not highest similairty score
# df_smltry_selmax_p = df_smltry_all.groupby("id_valid").apply(lambda x: x.loc[x["impact_sim_identical"].idxmax()]) # 52 cases
# df_smltry_selmax_p = df_smltry_all.groupby("id_pred").apply(lambda x: x.loc[x["impact_sim_identical"].idxmax()]) # 175 cases
df_smltry_selmax_p.reset_index(drop=True, inplace=True)

## FIXME  df_smltry_all.groupby("id_valid"): should it has duplicated cases of id_pred ? - i dont think so! 
#  bc it means that there model missed cases in valid_set
## --> so all duplicated cases (except one-this is TP or FP) are actual FNs
print(df_smltry_selmax_p.info())
print(df_smltry_selmax_p.id_pred.nunique()  )  # should be len of df
print(df_smltry_selmax_p.duplicated().sum())


# FNs: extracts all duplicates (except first occurrence eg. id_Pred==537 occurs in df_smltry_selmax_p three times (1st case: TP or FP, 2nd and 3rd are FNs)
fns = df_smltry_selmax_p[df_smltry_selmax_p.duplicated(subset="id_pred", keep="first")]

print(fns.info())
fns.id_pred.value_counts()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   impact_pred           52 non-null     object 
 1   impact_valid          52 non-null     object 
 2   impact_sim_identical  52 non-null     int64  
 3   impact_sim_cos        52 non-null     float64
 4   impact_sim_pr         0 non-null      float64
 5   tp_tn_fp_fn           52 non-null     object 
 6   citation              52 non-null     object 
 7   chunk_text_pred       52 non-null     object 
 8   sentence_text_valid   52 non-null     object 
 9   identifier_valid      52 non-null     object 
 10  id_pred               52 non-null     int64  
 11  id_valid              52 non-null     int64  
dtypes: float64(2), int64(3), object(7)
memory usage: 5.0+ KB
None
22
0


In [ ]:
# return all cases which has max sim also when max score is shared by multiple rows 
# df_smltry_all.loc[df_smltry_all.groupby("id_valid").transform(lambda x: x==x.max()).astype('bool')].shape
mask = df_smltry_all.loc[df_smltry_all.id_valid==37].groupby("id_valid").apply(lambda x: x==x["impact_sim_identical"].max())
mask = mask.droplevel(0)
#.transform(lambda x: x==x.max())
tt = df_smltry_all.loc[df_smltry_all.id_valid==37]#
tt = tt.where(mask.impact_sim_identical==mask.impact_sim_identical.max()).dropna(how="all") # drop cases which have not highest similairty score

# tt[mask]

# would return only first case of max sim:
#df_smltry_all.loc[df_smltry_all.id_valid==37].groupby("id_valid").apply(lambda x: x.loc[x["impact_sim_identical"].idxmax()]) #


In [ ]:
# FPs. 
print("False alarms (where model predicted ci but no corresponding valid case exists)", 
      len(df_pred["infrastructure_type"])  - len(df_smltry_selmax[df_smltry_selmax["impact_sim_identical"]== 1, "ci_group_pred"])
    )
# Get FPs - cases where model predicted presence of CI (but actually it is absent in valid set)
tt = df_pred.merge(
    # FIXME issue that df_pred_valid_all contains some duplicates where id_pred identical but not valid_entries
    df_smltry_selmax.drop_duplicates(), # safety: make sure that merging is done on 1:1 match
    left_on="id_pred",#["citation_id", "chunk_id","infrastructure_type", "damage", "location"], 
    right_on="id_pred",#["citation_id", "chunk_id_pred", "ci_pred", "damage_pred", "location_pred"],
    how="left",
    indicator=True    # return an extra column indicating which table the row was from.
)
tt = tt.loc[tt["_merge"] == "left_only"].drop(columns=["_merge"])
print("False positives (model predicted CI but no corresponding valid case exists):", len(tt))

607
False positives (model predicted CI but no corresponding valid case exists): 986


In [34]:
print(df_pred.shape[0])
# print(df_pred_valid_all.info())
print(tt.info())

1161
<class 'pandas.core.frame.DataFrame'>
Index: 986 entries, 75 to 1539
Data columns (total 25 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   citation_id_x         986 non-null    object 
 1   chunk_id              986 non-null    int64  
 2   infrastructure_type   986 non-null    object 
 3   damage                986 non-null    object 
 4   location              986 non-null    object 
 5   ci_entity             38 non-null     object 
 6   geo_entity            38 non-null     object 
 7   case_type             0 non-null      float64
 8   chunk_text            986 non-null    object 
 9   id_pred               986 non-null    int64  
 10  infrastructure_group  986 non-null    object 
 11  citation_id_y         0 non-null      object 
 12  ci_pred               0 non-null      object 
 13  ci_group_pred         0 non-null      object 
 14  damage_pred           0 non-null      object 
 15  location_pred        

In [185]:
df_pred#["infrastructure_type"]

,citation_id,chunk_id,infrastructure_type,damage,location,ci_entity,geo_entity,case_type,chunk_text
0,Karakatsani 2023,0,roads,severely damaged,Thessaly plain,National roads,plain,NaN,"The economic impact of the recent devastating floods in Greece The economic impact of the recent devastating floods in Greece The briefing presents the economic impact of the damages caused by the storm “Daniel”. The floods caused by the heavy rainfall, especially in Thessaly plain -accounting for approximately 15% of Greece’s agricultural land- resulted to a massive destruction in agriculture, infrastructure and residences, which is expected to negatively affect the Greek economy in short and mid-term. Fears for food shortages and increase of product prices have been raised. In addition, increase of fiscal costs, imports and unemployment may also affect the economy in the upcoming years. The extent of the impact to the economy is not yet know. Before the country recovered from the wildfires of August and the consequent huge forest disaster, a weather stormy phenomenon named Daniel hit the country. Specifically, in the beginning of September, rainfall of unprecedented intensity fell mainly in central Greece and especially in Thessaly plain, causing floods throughout the territory. Thousands of acres of crops and livestock farms were destroyed. Properties and houses were lost under ton of waters. National roads were closed, bridges collapsed, and some parts of the railway network were highly damaged dividing the country in two. Evidently, the massive destruction of agricultural"
1,Karakatsani 2023,0,bridges,collapsed,central Greece,NaN,NaN,NaN,"The economic impact of the recent devastating floods in Greece The economic impact of the recent devastating floods in Greece The briefing presents the economic impact of the damages caused by the storm “Daniel”. The floods caused by the heavy rainfall, especially in Thessaly plain -accounting for approximately 15% of Greece’s agricultural land- resulted to a massive destruction in agriculture, infrastructure and residences, which is expected to negatively affect the Greek economy in short and mid-term. Fears for food shortages and increase of product prices have been raised. In addition, increase of fiscal costs, imports and unemployment may also affect the economy in the upcoming years. The extent of the impact to the economy is not yet know. Before the country recovered from the wildfires of August and the consequent huge forest disaster, a weather stormy phenomenon named Daniel hit the country. Specifically, in the beginning of September, rainfall of unprecedented intensity fell mainly in central Greece and especially in Thessaly plain, causing floods throughout the territory. Thousands of acres of crops and livestock farms were destroyed. Properties and houses were lost under ton of waters. National roads were closed, bridges collapsed, and some parts of the railway network were highly damaged dividing the country in two. Evidently, the massive destruction of agricultural"
2,Karakatsani 2023,0,railway,highly damaged,Thessaly plain,NaN,NaN,NaN,"The economic impact of the recent devastating floods in Greece The economic impact of the recent devastating floods in Greece The briefing presents the economic impact of the damages caused by the storm “Daniel”. The floods caused by the heavy rainfall, especially in Thessaly plain -accounting for approximately 15% of Greece’s agricultural land- resulted to a massive destruction in agriculture, infrastructure and residences, which is expected to negatively affect the Greek economy in short and mid-term. Fears for food shortages and increase of product prices have been raised. In addition, increase of fiscal costs, imports and unemployment may also affect the economy in the upcoming years. The extent of the impact to the economy is not yet know. Before the country recovered from the wildfires of August and the consequent huge forest disaster, a weather stormy phe

In [ ]:
df_smltry_selmax.info()

<class 'pandas.core.frame.DataFrame'>
Index: 41 entries, education_school_damaged_Rhineland-Palantine to water_supply_little to no_Netherlands
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   impact_valid         41 non-null     object 
 1   impact_pred          41 non-null     object 
 2   impact_sim           41 non-null     float64
 3   impact_pr_sim        41 non-null     int64  
 4   tp_tn_fp_fn          41 non-null     object 
 5   citation             41 non-null     object 
 6   chunk_text_pred      41 non-null     object 
 7   sentence_text_valid  41 non-null     object 
 8   identifier_valid     41 non-null     object 
dtypes: float64(1), int64(1), object(7)
memory usage: 4.2+ KB


In [166]:
# df_smltry_selmax["impact_sim_identical"] < similarity_threshold

In [136]:
# len(df_valid_pred_same_docs["ci1_group"]) 

In [ ]:
# TODO fix FNs
print(df_valid_pred_same_docs.info())
print(df_smltry_selmax[df_smltry_selmax["impact_sim_identical"]==1].info())
# --> FNS should be  23
len(df_valid_pred_same_docs["ci1_group"])  - len(df_smltry_selmax[df_smltry_selmax["impact_sim_identical"]==1]["impact_valid"])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68 entries, 0 to 67
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   publication_id      68 non-null     object
 1   sentence_reference  68 non-null     object
 2   ci1_type            68 non-null     object
 3   ci1_damage          62 non-null     object
 4   ci1_location        55 non-null     object
 5   ci1_group           68 non-null     object
 6   identifier_valid    68 non-null     object
dtypes: object(7)
memory usage: 3.8+ KB
None
<class 'pandas.core.frame.DataFrame'>
Index: 45 entries, airports_affected_Malaga area to water_supply_little to no_Netherlands
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   impact_valid          45 non-null     object 
 1   impact_pred           45 non-null     object 
 2   impact_sim_identical  45 non-null     int64  
 3   

In [ ]:
# #df_valid_pred_same_docs["id_valid"] = df_valid_pred_same_docs.apply(lambda x: f"{x['ci1_group']}_{x['ci1_damage']}_{x['ci1_location']}_{x['sentence_text_valid'][:50]}", axis=1)
# print(df_valid_pred_same_docs["id_valid"].unique().__len__())
# print(df_valid_pred_same_docs.shape[0])
# ## --> check why electricity_others_outages_nan = 3  - (seems correct as sentences_ref are diff). airports_affected_Malaga area=2 are not unique
# df_valid_pred_same_docs[df_valid_pred_same_docs["id_valid"] == "airports_affected_Malaga area"]

# Improve similarity calculation
As all similarity measures - nomatter which emebdding model or kind of cosine similarity measure) were not sufficient eg. port ~ power to similar to port~harbor

Thus, it might be better to first group ci impacts into subgroups e.g .based on HARCI-EU categories,as some kind of postprocessing step before applying the similarity measurements



In [ ]:
df_ner = pd.read_json("./ner_patterns.jsonl/patterns", lines=True)

,label,pattern,subgroup_name,subgroup_name_note
0,CI_TYPE,[{'TEXT': {'REGEX': '.*[Rr]oad.*?'}}],road_others,unsure -if whitespace at beginning correct
1,CI_TYPE,[{'TEXT': {'REGEX': '.*[Aa]ccess road.*?'}}],road_others,NaN
2,CI_TYPE,[{'TEXT': {'REGEX': '.*[Hh]ighway.*'}}],motorways,NaN
3,CI_TYPE,[{'TEXT': {'REGEX': '.*[Mm]otorway.*'}}],motorways,NaN
4,CI_TYPE,[{'TEXT': {'REGEX': '.*[Rr]ail.*'}}],rail,NaN
...,...,...,...,...
97,CI_TYPE,[{'TEXT': {'REGEX': '.*[Mm]edical.*'}}],healthcare_others,NaN
98,CI_TYPE,[{'TEXT': {'REGEX': '.*[Cc]linic.*'}}],healthcare_hospitals_clinics,NaN
99,CI_TYPE,[{'TEXT': {'REGEX': '.*[Hh]ospital.*'}}],healthcare_hospitals_clinics,NaN
100,CI_TYPE,[{'TEXT': {'REGEX': '.*[Ee]mergency.*'}}],healthcare_hospitals_clinics,NaN


In [ ]:
# d = {"label":"CI_TYPE","pattern": [{"TEXT": {"REGEX": ".* [Tt]ransport.*"}}, {"LOWER": "sector"}], "subgroup_name":"transport_others"}
# #d["pattern"][0]["TEXT"]["REGEX"] + " " + dd["pattern"][1]["LOWER"]
# d["subgroup_name"]

'transport_others'

In [ ]:
import re

# load regular expressions and subgroups from NER patterns as dict
# for general cases and all special cases with "LOWER"-pattern
regexes_1 = [
        {df_ner["pattern"][i][0]["TEXT"]["REGEX"] : df_ner["subgroup_name"][i]}
          for i in range(len(df_ner)) 
            if len(df_ner["pattern"][i])==1 
]
regexes_2 = [
    {df_ner["pattern"][i][0]["TEXT"]["REGEX"] + " " + df_ner["pattern"][i][1]["LOWER"] : df_ner["subgroup_name"][i] }
      for i in range(len(df_ner))
        if len(df_ner["pattern"][i])==2
]
# regexes = regexes_1 | regexes_2
regexes = regexes_1 + regexes_2
regexes[:20]


[{'.*[Rr]oad.*?': 'road_others'},
 {'.*[Aa]ccess road.*?': 'road_others'},
 {'.*[Hh]ighway.*': 'motorways'},
 {'.*[Mm]otorway.*': 'motorways'},
 {'.*[Rr]ail.*': 'rail'},
 {'.*[Rr]ailway track.*?': 'rail'},
 {'.*[Ss]tation.*': 'railway_station'},
 {'.*[Mm]etro station.*': 'metro'},
 {'.*[Bb]ridge.*?': 'bridges'},
 {'.*[Aa]irport.*': 'airports'},
 {'.*[Aa]viation.*': 'aviation'},
 {'.*[Aa]viation industry': 'aviation'},
 {'.*[Ss]ea port.*': 'ports'},
 {'.*[Pp]ort.*': 'ports'},
 {'.*[Hh]arbor.*': 'ports'},
 {'.*[Mm]aritime sector': 'maritime'},
 {'.*[Ww]aterway.*': 'inland_waterways'},
 {'.*[Ii]nland waterway.*?': 'inland_waterways'},
 {'.*IWW.*?': 'inland_waterways'},
 {'.*[Ee]lectric.* supply': 'electricity_supply'},
 {'.*[Ee]lectric.* net.*': 'electricity_distribution'},
 {'.*[Ee]lectric.* plant.*': 'powerplants'},
 {'.*[Ee]lectric.* service.*': 'electricity_supply'},
 {'.*[Pp]ower net.*': 'electricity_distribution'},
 {'.*[Pp]ower supply': 'electricity_supply'},
 {'.*[Gg]as pipe.*': '

In [ ]:
for i, r in enumerate(regexes):

    # get regex pattern for CI type (key) and its subgroup (value)
    # NOTE, nice shortcut: get key containing regex by unpacking each dict into list, then get key
    pattern = [*r][0]
    subgroup = r[pattern]

    # assign subgroups to CI records, na=False to remove all records which not match patterns
    mask = df_pred["infrastructure_type"].str.contains(pattern, regex=True, na=False)
    df_pred.loc[mask, "infrastructure_group"] = subgroup

    # assign subgroups to CI records, na=False to remove all records which not match patterns
    mask = df_valid["ci1_type"].str.contains(pattern, regex=True, na=False)
    df_valid.loc[mask, "ci1_group"] = subgroup
    
df_pred

print(df_pred.infrastructure_group.isna().sum())
print(df_pred.infrastructure_group.value_counts())
# df_pred.infrastructure_group.unique()


,citation_id,chunk_id,infrastructure_type,damage,location,ci_entity,geo_entity,case_type,chunk_text,infrastructure_type_postp,infrastructure_type_postp2,infrastructure_group
0,Karakatsani 2023,0,roads,severely damaged,Thessaly plain,National roads,plain,NaN,"The economic impact of the recent devastating floods in Greece The economic impact of the recent devastating floods in Greece The briefing presents the economic impact of the damages caused by the storm “Daniel”. The floods caused by the heavy rainfall, especially in Thessaly plain -accounting for approximately 15% of Greece’s agricultural land- resulted to a massive destruction in agriculture, infrastructure and residences, which is expected to negatively affect the Greek economy in short and mid-term. Fears for food shortages and increase of product prices have been raised. In addition, increase of fiscal costs, imports and unemployment may also affect the economy in the upcoming years. The extent of the impact to the economy is not yet know. Before the country recovered from the wildfires of August and the consequent huge forest disaster, a weather stormy phenomenon named Daniel hit the country. Specifically, in the beginning of September, rainfall of unprecedented intensity fell mainly in central Greece and especially in Thessaly plain, causing floods throughout the territory. Thousands of acres of crops and livestock farms were destroyed. Properties and houses were lost under ton of waters. National roads were closed, bridges collapsed, and some parts of the railway network were highly damaged dividing the country in two. Evidently, the massive destruction of agricultural",road_others,roads,road_others
1,Karakatsani 2023,0,bridges,collapsed,central Greece,NaN,NaN,NaN,"The economic impact of the recent devastating floods in Greece The economic impact of the recent devastating floods in Greece The briefing presents the economic impact of the damages caused by the storm “Daniel”. The floods caused by the heavy rainfall, especially in Thessaly plain -accounting for approximately 15% of Greece’s agricultural land- resulted to a massive destruction in agriculture, infrastructure and residences, which is expected to negatively affect the Greek economy in short and mid-term. Fears for food shortages and increase of product prices have been raised. In addition, increase of fiscal costs, imports and unemployment may also affect the economy in the upcoming years. The extent of the impact to the economy is not yet know. Before the country recovered from the wildfires of August and the consequent huge forest disaster, a weather stormy phenomenon named Daniel hit the country. Specifically, in the beginning of September, rainfall of unprecedented intensity fell mainly in central Greece and especially in Thessaly plain, causing floods throughout the territory. Thousands of acres of crops and livestock farms were destroyed. Properties and houses were lost under ton of waters. National roads were closed, bridges collapsed, and some parts of the railway network were highly damaged dividing the country in two. Evidently, the massive destruction of agricultural",bridges,bridges,bridges
2,Karakatsani 2023,0,railway,highly damaged,Thessaly plain,NaN,NaN,NaN,"The economic impact of the recent devastating floods in Greece The economic impact of the recent devastating floods in Greece The briefing presents the economic impact of the damages caused by the storm “Daniel”. The floods caused by the heavy rainfall, especially in Thessaly plain -accounting for approximately 15% of Greece’s agricultural land- resulted to a massive destruction in agriculture, infrastructure and residences, which is expected to negatively affect the Greek economy in short and mid-term. Fears for food shortages and increase of product prices have been raised. In addition, increase of fiscal costs, imports and unemployment may also affect the economy in the upcoming years. The extent of the impact to the economy is not y

In [ ]:
print(df_valid.df_valid.isna().sum())
print(df_valid.df_valid.value_counts())
# df_pred.infrastructure_group.unique()

array(['ports', 'road_others', 'electricity_others', 'motorways', nan,
       'rail', 'bridges', 'gas_distribution', 'waste_others',
       'wastewater', 'water_others', 'drinking_water', 'water_supply',
       'it_telecommunication', 'healthcare_others',
       'healthcare_hospitals_clinics', 'education_school'], dtype=object)

In [ ]:
# s = "dyke" to s2 = "levee", s3 = "dam"
# bge-m3: 0.48  0.54
# all-MIniLM-L6-v2: 0.34 , 0.36  (similar all-mpnet-base-v2)
# gensim word2vec: 0.39 0.40


# s1 = "aviation" s2 = "air traffic"
# word vector spacy: 0.45
# contextual vector spacy: 0.68
# bge-m3: 0.76
# all-MIniLM-L6-v2: xx  (all-mpnet-base-v2: 0.79)
# gensim word2vec: 


# s1 = "power" s2 = "electricity"
# word vector spacy: 0.61
# contextual vector spacy: 0.66
# bge-m3: 
# all-MIniLM-L6-v2: xx   (all-mpnet-base-v2: 0.43)
# gensim word2vec: 0.58


# s1 = "electricity infrastructure" s2 = "electricity"
# word vector spacy: 0.87
# contextual vector spacy: 0.71
# bge-m3: 
# all-MIniLM-L6-v2:   xx  (all-mpnet-base-v2: 0.63)
# gensim word2vec: 


# s1 = "transportation" s2 = "transport infrastructure"
# word vector spacy: 
# contextual vector spacy: 
# bge-m3: 
# all-MIniLM-L6-v2:   xx  (all-mpnet-base-v2: 0.84)
# gensim word2vec: 

# s1 = "port" s2 = "power"  s3= harbour
# bge-m3: 0.58, 0.50
# all-MIniLM-L6-v2: 0.33 , 0.56  (similar all-mpnet-base-v2)
# gensim word2vec: 0.14 0.59

# s1 = "electricity" s2 = "transportation" 
# bge-m3:  0.64
# all-MIniLM-L6-v2:   (all-mpnet-base-v2: 0.47)
# gensim word2vec: 0.33

tensor([[0.5883]])

In [ ]:
# # print(cos_sim(model_scs["transportation"], model_scs["transport infrastructure"]))
# # print(cos_sim(model_scs["electricity infrastructure"], model_scs["electricity"]))
# # print(cos_sim(model_scs["power plant"], model_scs["electricity"]))
# print(cos_sim(model_scs["power"], model_scs["electricity"]))
# print(cos_sim(model_scs["aviation"], model_scs["air traffic"]))
# # identical to model_scs.similarity("port", "power"))


# # similarity_score = 1-distance.cosine(model.encode([s1])[0], model.encode([s2])[0])

### Analyse evaluation results 


In [ ]:
df_smltry_selmax#.info()

In [ ]:
## find out for which docs model performed bad (or good)
## based on this info try to improve model 

df_smltry_selmax.dropna(subset=["impact_sim_cos"]).groupby("citation").apply(lambda x: x.loc[x["impact_sim_cos"].idxmax()]).sort_values(by="impact_sim_cos", ascending=True)
## check EFE, Wilson, European Investment Bank, Containerlift, Lloyds List, Gilbody Dickerson


In [ ]:
## check entries of worst performace docs for damage
df_smltry_selmax.loc[df_smltry_selmax["citation"].isin(["Khazai 2023", "ABC 2024", "Containerlift 2024", "Lloyds List 2024", "Ferlita 2023"])]

In [ ]:
## check entries of worst performance docs for Ci tyes
df_smltry_selmax.loc[df_smltry_selmax["citation"].isin(["EFE 2024", "Containerlift 2024", "Lloyds List 2024", "Wilson 2024", "Gilbody Dickerson 2024", "European Investment Bank 2025"])].head(50)


## For each validation entry, search for all prediction cases of the same chunk 

In [ ]:
## get same impact entries
columns_valid = ["ci1_type", "ci1_damage", "ci1_location"]
columns_pred = ["infrastructure_type", "damage", "location"]


for column_valid, column_pred in zip(columns_valid, columns_pred):

    print(f" --------- Processing column pair: {column_valid} - {column_pred} ------------")
    
    df_valid_pred_all = pd.DataFrame()
    citations_list = []

    ## for each validation record
    for i in range(len(df_valid)):
        
        highest_similarity_score = 0.00
        
        ## needed to traceback info when entry is missing in pred. DS
        # chunk_id_value_valid = df_valid.chunk_id[i]

        # select nth validation record and check that it has value
        df_valid_entry = df_valid.iloc[i]
        if df_valid_entry[column_valid] is np.nan:
            continue
        
        citation_str = df_valid_entry.publication_id
        citations_list.append(citation_str)


        # get all corresponding prediction records
        df_pred_entries = df_pred[df_pred["citation_id"].isin([citation_str])]

        #  handle on NANs
        df_pred_entries[column_pred] = np.where(df_pred_entries[column_pred].isna(), "nan", df_pred_entries[column_pred])
        # df_pred_entries[column_pred] = df_pred_entries[column_pred].astype(str)
        # remove double whitespaces
        # df_pred_doc[column_pred] = df_pred_doc[column_pred].replace("  ", " ")
        # df_valid_entries[column_valid] = df_valid_entries[column_valid].replace("  ", " ")


        # vector of validiation entry 
        valid_impact = df_valid_entry[column_valid]
        valid_vec = nlp(valid_impact).vector

        # print(" ------- Searching for citation:", citation_str, " in predictions ------- ")

        # Compute similarity between each validation CI impact case and all potential predicted CI impact cases (cross-product)
        for j in range(len(df_pred_entries[column_pred])):

            if df_pred_entries[column_pred].iloc[j] == "nan":
                continue

            pred_impact = df_pred_entries[column_pred].iloc[j]

            pred_vec = nlp(pred_impact).vector
            similarity_score = cosine_similarity(valid_vec, pred_vec)  # 0-1 value, the higher the more similar
            # print(f"Similarity {i}-{j}: {similarity_score}")

            # print(f"Searching for highest similarity ... ")
            ## get only pair with highest similarity
            if similarity_score > highest_similarity_score:
                
                highest_similarity_score = similarity_score
                
                dict_pair = {
                    "impact_valid": valid_impact, 
                    "impact_pred": pred_impact, 
                    "similarity": highest_similarity_score,
                    "citation": citation_str,
                    "chunk_id_pred": (df_pred.chunk_id[i],  df_pred.chunk_id[j])
                }
            else:
                continue

        df_valid_pred_all = pd.concat([df_valid_pred_all, pd.DataFrame([dict_pair])], ignore_index=True)


    print(f" ---------- Evaluation summary statistics - {column_pred}: -----------")
    print(df_valid_pred_all.similarity.describe())

    

    SIMILARITY_FILENAME = f'{column_pred}_{SIMILARITY_LLM_FILENAME}'
    SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

    print("Saving evaluation statistics, distribution plots, and scores to ", SIMILARITY_FILEPATH.stem, "[.parquet, _stats.json]")
    with open(SIMILARITY_FILEPATH, 'w') as f:
        # results
        df_valid_pred_all.to_csv(SIMILARITY_FILEPATH.with_suffix('.csv'), index=False)
        df_valid_pred_all_pyarrow = pa.Table.from_pandas(df_valid_pred_all)
        pq.write_table(df_valid_pred_all_pyarrow, SIMILARITY_FILEPATH)   
        #   summary statistics
        df_valid_pred_all_stats = df_valid_pred_all.describe()
        f = SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_stats.json"  
        df_valid_pred_all_stats.to_json(f, indent=4)
        # distribution plots
        df_valid_pred_all.similarity.hist(bins=100).to_file(SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_hist.png")



    # similarity_threshold = 0.75
    # df_valid_pred_all['is_similar'] = df_valid_pred_all['similarity'] <= similarity_threshold
    # print(f"Number of similar impact cases (similarity >= {similarity_threshold}): {df_valid_pred_all['is_similar'].sum()} out of {len(df_valid_pred_all)}\n")

    # df_valid_pred_all =  df_valid_pred_all[df_valid_pred_all['similarity'] <= similarity_threshold]

    # SIMILARITY_FILENAME = f'{column_pred}_lower75_{SIMILARITY_LLM_FILENAME}'
    # SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

    # with open(SIMILARITY_FILEPATH, 'w') as f:
    #     # results
    #     df_valid_pred_all.to_csv(SIMILARITY_FILEPATH.with_suffix('.csv'), index=False)
    #     df_valid_pred_all_pyarrow = pa.Table.from_pandas(df_valid_pred_all)
    #     pq.write_table(df_valid_pred_all_pyarrow, SIMILARITY_FILEPATH)  
    #     #   summary statistics
    #     df_valid_pred_all_stats = df_valid_pred_all.describe()
    #     f = SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_stats.json"  
    #     df_valid_pred_all_stats.to_json(f, indent=4)


In [ ]:
# df_valid_pred_all[df_valid_pred_all['similarity'] <= 0.75]

# df_valid_pred_all.similarity.hist(bins=100)

In [ ]:
columns_pred

In [ ]:
LLM_DATA_FILEPATH

### Load parquet file

In [ ]:

columns_pred = ["infrastructure_type", "damage", "location"]

In [ ]:
column_pred = "infrastructure_type"
SIMILARITY_FILENAME = f'llm1_similarity_{column_pred}_75.parquet'
SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    df = pd.read_parquet(SIMILARITY_FILEPATH, engine='pyarrow')
    display(df)

## Archive

In [ ]:
## Aim 
## for all identical valid entries ie. with same [ci_valid	damage_valid	location_valid	sentence_text_valid]
## get the match to pred_entity with highest similarity

In [ ]:
    # ## calc for each entry with the same chunk_text the similarity between valid_impact and pred_impact
    # ## means we calc also the False Negatives (ie. where valid entry exists but no prediction)


    # # iterate over groups of entities which refer to the same valid case (i.e. which are identical in valid_columns)
    # # TODO iterate over unqiue cases in df_valid (instead of using grouper)
    # grouper = df_pred_valid_all[["ci_valid", "damage_valid", "location_valid", "sentence_text_valid"]].drop_duplicates()
    # for group in grouper.itertuples():
    #     df_pred_valid_group = df_pred_valid_all[df_pred_valid_all[["ci_valid", "damage_valid", "location_valid", "sentence_text_valid"]] == group[["ci_valid", "damage_valid", "location_valid", "sentence_text_valid"]]]

    #     # calc. similarities to pred_entities
    #     for i, entry in df_pred_valid_group.iterrows():

    #         highest_similarity_score = 0 

    #         if entry[column_pred].iloc[i] == "nan":
    #             continue
            
    #         # calc embeddings
    #         pred_impact = entry[column_pred].iloc[i]
    #         pred_vec = nlp(pred_impact).vector

    #         valid_impact = entry[column_valid].iloc[i] # is unique for each group
    #         valid_vec = nlp(valid_impact).vector
    #         print(valid_impact, "valid_impact")
            
    #         similarity_score = cosine_similarity(valid_vec, pred_vec)  # 0-1 value, the higher the more similar
    #         # print(f"Similarity {i}-{j}: {similarity_score}")

    #         ## return only pred-valid-pair with highest similarity
    #         if similarity_score > highest_similarity_score:
                
    #             highest_similarity_score = similarity_score
                
    #             entry["impact_similarity"] = highest_similarity_score

    # ## FNs
    # # # calc FN when valid_info exists but not corresponding pred_info
    # ## number of FNs is small due that wrong matching with any chunk-text is more likely due to its text size comapred sentence-level (valid set) 
    # elif entry[column_pred] is np.nan:
    #     similarity_score = 0
    #     dict_pair = {
    #         "impact_valid": valid_impact, 
    #         "impact_pred": pred_impact, 
    #         "impact_similarity": similarity_score,
    #         "tp_tn_fp_fn": "fn",
    #         "citation": entry.citation_id,
    #         "chunk_text_pred": entry.chunk_text_pred,
    #         "sentence_text_valid": entry.sentence_text_valid,
    #         }
    #     df_smltry_selmax = pd.concat([df_smltry_selmax, pd.DataFrame([dict_pair])], ignore_index=True)

    # ## FPs
    # elif entry[column_valid] is np.nan:
    #     similarity_score = 0
    #     dict_pair = {
    #         "impact_valid": valid_impact, 
    #         "impact_pred": pred_impact, 
    #         "impact_similarity": similarity_score,
    #         "tp_tn_fp_fn": "fp",
    #         "citation": entry.citation_id,
    #         "chunk_text_pred": entry.chunk_text_pred,
    #         "sentence_text_valid": entry.sentence_text_valid,
    #         }
    #     df_smltry_selmax = pd.concat([df_smltry_selmax, pd.DataFrame([dict_pair])], ignore_index=True)




In [ ]:
# ## get same impact entries
# columns_valid = ["ci1_type", "ci1_damage", "ci1_location"]
# columns_pred = ["infrastructure_type", "damage", "location"]



## iterate over predictions and search for each prediction reocrds for corresponding valid cases 

# for column_valid, column_pred in zip(columns_valid, columns_pred):

#     print(f" --------- Processing column pair: {column_valid} - {column_pred} ------------")
    
#     df_valid_pred_all = pd.DataFrame()
#     citations_list = []

#     ## for each validation record
#     for i in range(len(df_valid)):
        
#         highest_similarity_score = 0.00
        
#         ## needed to traceback info when entry is missing in pred. DS
#         # chunk_id_value_valid = df_valid.chunk_id[i]

#         # select nth validation record
#         df_valid_entry = df_valid.iloc[i]
#         citation_str = df_valid_entry.publication_id
#         citations_list.append(citation_str)
#         print(" ------- Searching for citation:", citation_str, " in predictions ------- ")


#         # get all corresponding prediction records
#         df_pred_entries = df_pred[df_pred["citation_id"].isin([citation_str])]
#         #  handle on NANs
#         df_pred_entries[column_pred] = np.where(df_pred_entries[column_pred].isna(), "nan", df_pred_entries[column_pred])
#         # df_pred_entries[column_pred] = df_pred_entries[column_pred].astype(str)
#         # remove double whitespaces
#         # df_pred_doc[column_pred] = df_pred_doc[column_pred].replace("  ", " ")
#         # df_valid_entries[column_valid] = df_valid_entries[column_valid].replace("  ", " ")

#         # skip when validation entry ha no value
#         if df_valid_entry[column_valid] is np.nan:
#             continue

#         # vector of validiation entry 
#         valid_impact = df_valid_entry[column_valid]
#         valid_vec = nlp(valid_impact).vector


#         # Compute similarity between each predicted impact case and all potential validation impact cases (cross-product)
#         # print(f"Searching for highest similarity of`{pred_impact}` in validation set ... ")
#         for j in range(len(df_pred_entries[column_pred])):

#             if df_pred_entries[column_pred].iloc[j] == "nan":
#                 continue

#             pred_impact = df_pred_entries[column_pred].iloc[j]

#             pred_vec = nlp(pred_impact).vector
#             similarity_score = cosine_similarity(valid_vec, pred_vec)  # 0-1 value, the higher the more similar
#             # print(f"Similarity {i}-{j}: {similarity_score}")

#             ## get only pair with highest similarity
#             if similarity_score > highest_similarity_score:
                
#                 highest_similarity_score = similarity_score
                
#                 dict_pair = {
#                     "impact_valid": valid_impact, 
#                     "impact_pred": pred_impact, 
#                     "similarity": highest_similarity_score,
#                     "citation": citation_str,
#                     "chunk_id_pred": df_pred.chunk_id[i]
#                 }
#             else:
#                 continue

#         df_valid_pred_all = pd.concat([df_valid_pred_all, pd.DataFrame([dict_pair])], ignore_index=True)


#     print(" ---------- Evaluation summary statistics: -----------")
#     print(df_valid_pred_all.similarity.describe())



#     SIMILARITY_FILENAME = f'{SIMILARITY_LLM_FILENAME}_{column_pred}.parquet'
#     SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

#     print("Saving evaluation statistics and scores to ", SIMILARITY_FILEPATH.stem, "[.parquet, _stats.json]")
#     with open(SIMILARITY_FILEPATH, 'w') as f:
#         # results
#         df_valid_pred_all_pyarrow = pa.Table.from_pandas(df_valid_pred_all)
#         pq.write_table(df_valid_pred_all_pyarrow, SIMILARITY_FILEPATH)   
#         #   summary statistics
#         df_valid_pred_all_stats = df_valid_pred_all.describe()
#         f = SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_stats.json"  
#         df_valid_pred_all_stats.to_json(f, indent=4)



#     similarity_threshold = 0.75
#     df_valid_pred_all['is_similar'] = df_valid_pred_all['similarity'] >= similarity_threshold
#     print(f"Number of similar impact cases (similarity >= {similarity_threshold}): {df_valid_pred_all['is_similar'].sum()} out of {len(df_valid_pred_all)}")

#     SIMILARITY_FILENAME = f'{SIMILARITY_LLM_FILENAME}_{column_pred}_75.parquet'
#     SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

#     with open(SIMILARITY_FILEPATH, 'w') as f:
#         # results
#         df_valid_pred_all_pyarrow = pa.Table.from_pandas(df_valid_pred_all)
#         pq.write_table(df_valid_pred_all_pyarrow, SIMILARITY_FILEPATH)  
#         #   summary statistics
#         df_valid_pred_all_stats = df_valid_pred_all.describe()
#         f = SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_stats.json"  
#         df_valid_pred_all_stats.to_json(f, indent=4)


In [ ]:

# #  Define folder for handling and writing outputs
# def write_to_file(data, out_folder, filename):
#     """Convert output to DataFrame and write to file"""
#     df = pd.DataFrame(list(data), columns=['tag', 'sts_score'])
#     #  Sort the DataFrame by similarity (explicitly)
#     df = df.sort_values(by='sts_score', ascending=False)
#     #  Assign integers to ranking
#     df['rank'] = df['sts_score'].rank(method='first', ascending=False).astype(int)
#     #  Only keep the first 20 resulting tags
#     df = df.head(50)
#     #  Save to file
#     df.to_csv(out_folder / f'{filename}_output.csv', index=False)

# #  Fill run metrics to dictionary
# def handle_metrics(metrics, model_name, length, end_time, start_time):
#     print(f'-> Took {end_time - start_time:.2f} seconds. Number of tags: {length}.')
#     metrics.append({
#         'modelname': model_name,
#         'runtime': round(end_time - start_time, 2),
#         'tagcount': length
#     })
#     return metrics

# class CPU_Unpickler(pickle.Unpickler):
#     """Fix for having issues with loading models on CPU"""
#     def find_class(self, module, name):
#         if module == 'torch.storage' and name == '_load_from_bytes':
#             return lambda b: torch.load(io.BytesIO(b), map_location='cpu')
#         else: return super().find_class(module, name)
